In [34]:
from concurrent.futures import ThreadPoolExecutor

import geopandas as gpd
import numpy as np
import odc.geo
from odc.stac import load
from pystac_client import Client
from shapely import geometry
from sklearn.ensemble import RandomForestClassifier

%matplotlib inline

import datacube
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

import sys, os
sys.path.insert(1, os.path.abspath('../Tools'))

from pystac.client import Client


import rasterio
import matplotlib
import pickle
import joblib
from utils import load_data, predict_xr, safe_divide

In [35]:
%reload_ext autoreload
%autoreload 2

## Find and load data

Load data and set up your array to use for prediction

In [38]:
# Configure some things up front
chunks = dict(x=2048, y=2048)
datetime = "2022"

bbox = [177.24, -17.95, 177.70, -17.58] #nadi# 
# bbox = [178.36683199429106, -17.706311982643278, 178.54589121502428, -17.561129121023423]
# bbox = [177.2, -18.3, 178.7, -17.3]

bbox_geometry = geometry.box(*bbox)

gdf = gpd.GeoDataFrame({'geometry': [bbox_geometry]}, crs='EPSG:4326')
gdf.explore()

In [47]:
catalog = "https://stac.digitalearthpacific.org"
client = Client.open(catalog)

items = list(
    client.search(
        collections=["dep_s2s1_mrd"],
        bbox=bbox,
    ).items()
)


In [49]:
import pystac
from shapely.geometry import box

# Point directly to the collection.json or catalog.json inside that S3 prefix
stac_url = "https://stac.digitalearthpacific.org"
catalog = pystac.read_file(stac_url)

# Define your target bounding box as a Shapely polygon
aoi = box(*bbox)  # bbox = (min_lon, min_lat, max_lon, max_lat)

# Filter items manually by year and bounding box
matching_items = []
for item in catalog.get_all_items():
  if item.datetime and str(item.datetime.year) == "2022":
    if item.bbox and box(*item.bbox).intersects(aoi):
      matching_items.append(item)

print(f"Found {len(matching_items)} items")


Found 0 items


In [54]:
import os

# 1. Point AWS config away from the broken ~/.aws/config profile
os.environ["AWS_CONFIG_FILE"] = "/dev/null"
os.environ["AWS_SHARED_CREDENTIALS_FILE"] = "/dev/null"
os.environ["AWS_EC2_METADATA_DISABLED"] = "true"

import fsspec
import pystac
from shapely.geometry import box

# 2. Force a fresh anonymous S3 filesystem (bypasses fsspec's cache)
fs = fsspec.filesystem("s3", anon=True, skip_instance_cache=True)

# 3. List all JSON files in the folder
item_urls = fs.glob("dep-public-staging/dep_s2s1_mrd/0-0-2/**/*.json")
print(f"Total JSON files found: {len(item_urls)}")

# 4. Filter matching items by year and bounding box
aoi = box(*bbox)
matching_items = []

for path in item_urls:
  if path.endswith(("collection.json", "catalog.json")):
    continue
  # Load the STAC item directly via s3fs
  with fs.open(path, "r") as f:
    item = pystac.Item.from_dict(pystac.stac_io.json.load(f))

  if item.datetime and str(item.datetime.year) == "2022":
    if item.bbox and box(*item.bbox).intersects(aoi):
      matching_items.append(item)

print(f"Found {len(matching_items)} matching items")


Total JSON files found: 27
Found 6 matching items


In [41]:
# bbox_bounds = [178.36574794372646, -17.706311982643278, 178.5471181218782, -17.561129121023423]
# bbox_bounds = [177.2, -18.3, 178.7, -17.3]
bbox_bounds = [177.35, -17.8, 177.60, -17.5] #nadi


# Run the updated function
data = load_data(
    bbox=bbox_bounds,  
    # year="2024"  # Pass the target year seamlessly
)


In [56]:
print("Assets:", list(matching_items[0].assets.keys()))
print("CRS:", matching_items[0].properties.get("proj:epsg"))


Assets: ['class', 'proba']
CRS: None


In [57]:
import odc.stac
import rioxarray  # Enables the .rio accessor on xarray objects

# 1. Load and mosaic the 6 tiles clipped to your bbox
#    aws_unsigned=True prevents GDAL from using your broken ~/.aws/config SSO profile
ds = odc.stac.load(
    matching_items,
    bbox=bbox,  # (min_lon, min_lat, max_lon, max_lat) in EPSG:4326
    chunks={"x": 2048, "y": 2048},
    aws={"aws_unsigned": True},
)

# 2. If multiple timestamps exist for 2022, select one or compute a median composite
if "time" in ds.dims and ds.sizes["time"] > 1:
  ds_single = ds.median(dim="time", keep_attrs=True)
else:
  ds_single = ds.squeeze("time", drop=True)

# 3. Export a single band (e.g., replace "red" with your desired asset name)
#    OR stack all bands into a multi-band GeoTIFF:
multi_band = ds_single.to_array(dim="class")
multi_band.rio.to_raster("dep_s2s1_mrd_2022_clip.tif", compress="LZW")

print("Saved dep_s2s1_mrd_2022_clip.tif")


Saved dep_s2s1_mrd_2022_clip.tif


In [21]:
loaded_data = data.compute()